# SENTINEL — YOLOv8s Weapon FULL RETRAIN v4 on Kaggle (Phase 3, dataset v4)

**v3 recipe ka FULL RETRAIN, COCO yolov8s.pt se** (kisi purane model se fine-tune NAHI): 100 epochs, lr0 0.01, patience 20, Knife 2x oversampling. Dataset v4 = v3 train (24,146) MINUS audit fixes (897 Stage-1 exclusions + 139 Stage-2 drops + 25 flips + 26 adds + 5 rebox) PLUS 145 naye Test8/Test9 hard frames = 23,268 train. **2x T4** par andazan ~7-8 ghante.

**v4ft se seekha sabaq (21 Nov):** v4ft fine-tune (v3 se, 8 epochs, naye frames x10) ne clean test par 13 points ka nuqsan kiya (0.8378 -> 0.709) aur hard frames par bhi v3 se bura raha (F1 0.737 vs 0.860). Fine-tuning ka pattern teesri baar confirm hua (v2.1, v2.2, v4ft) — chhota naya set + replay hamesha clean accuracy khata hai. Full retrain hi woh form hai jo kaam karta hai (v3: clean +1.1 pts, hard F1 0.363 -> 0.860). v4 mein naye frames natural frequency par hain (0.6% — 100 epochs x 1 copy = 100 exposures, v4ft ke 80 ke barabar) taake model inhein seekhe baghair purana bhoole.

Dataset mix (script `scripts/data_preparation/assemble_v4_dataset.py` se bana): **23,123 clean v3 replay + 145 naye frames** = 23,268 train. Val 1,351 aur **test 680 FROZEN** (v2/v3 wali hi split) — naye weights sirf tab adopt honge jab frozen test par v3 baseline (mAP50 0.8666) ko hara dein AUR cleaned-test + hard-frame eval + live retest pass karein. v3 fallback rehta hai.

**Pehle ye 4 cheezein set karo (notebook settings, right sidebar):**
1. **Accelerator: GPU T4 x2** select karo (100-epoch run ke liye 2 GPUs LAZMI hain — 1 GPU par 12h session cap cross ho jayegi; Step 1 isay check karke fail ho jayega).
2. **Internet: ON** rakho (pip install + repo clone ke liye).
3. **Add Input → dataset `sentinel-weapon-v4`** attach karo (private dataset — review ho chuka hai, Kaggle par maujood hai).
4. **Starting weights COCO yolov8s.pt hain** (Ultralytics khud download karta hai) — Drive se kuch download nahi hota is run mein.

**Phir: Save & Run All** dabao. Kisi cell mai red error aye to **wahin ruk jao** aur output Muse ko bhejo, khud fix mat karo. Step 5 dataset ke `manifest.json` se review status check karta hai — labels UNREVIEWED hon to training wahin ruk jayegi.

Training script repo se ata hai (`scripts/training/train_weapon_detector.py`) — notebook use clone karke chalati hai, taake Kaggle aur repo hamesha ek hi code use karein.


In [ ]:
# ============ CONFIG — sirf yahi cell badalna hai ============
RUN_MODE = "v4"        # "v4" = 100-epoch full retrain COCO se | "smoke" = 3 epochs (pipeline check)
REPO_BRANCH = "main"   # training script main se clone hota hai
# REPO_COMMIT: jab set ho (poora ya chhota git SHA), Step 4 clone ke baad usi
# commit par checkout karta hai — floating main ki bajaye TESTED commit par
# pin karo (v5 ke liye). Khali ("") rakho to branch tip hi chalta hai.
# Jo commit bhi chale, uska SHA run_summary.json mein code_revision ke tor
# par record hota hai (Step 4 `git rev-parse HEAD` se).
REPO_COMMIT = ""
SEED = 42              # training seed — script ko --seed se explicit pass hota hai (v3 recipe; default par bharosa nahi)
OVERSAMPLE_KNIFE = 2  # v3 wala setting (2x) — saare Knife images par lagta hai (v3 recipe)
NEW_OVERSAMPLE = 1    # naye files ka oversampling BAND (1 = band) — full retrain mein 100 epochs x 1 copy = 100 exposures kaafi hain

from datetime import datetime
from pathlib import Path

import torch  # DEVICE/BATCH yahin derive hote hain (pehle Step 1 cell mein the)
_N_GPU_CONFIG = torch.cuda.device_count()
DEVICE = "0,1" if _N_GPU_CONFIG >= 2 else "0"   # 2 GPUs hon to DDP, 1 ho to single-GPU
BATCH = 32 if _N_GPU_CONFIG >= 2 else 16        # 16/GPU at imgsz 640 on 16 GB T4s

EPOCHS = 3 if RUN_MODE == "smoke" else 100
RUN_NAME = f"weapon_yolov8s_{RUN_MODE}_{datetime.now():%Y%m%d_%H%M}"

# ---- Full-retrain hyperparameters (v3 recipe — v3 ne prove kiya) ----
LR0 = 0.01           # v3 wala (fine-tune wala 0.001 NAHI — ye scratch training hai)
DEGREES = 20         # rotation +/-20 deg (research: 15-30; vertical-knife->gun failure ka fix)
PERSPECTIVE = 0.0005 # tilted camera (research: 0.0005-0.001)
CLOSE_MOSAIC = 10    # aakhri 10 epochs mosaic band (v3 wala)
MIXUP = 0.1          # research: cheap test (v2 mein 0.0 tha)
ERASING = 0.4        # occlusion (v2 default bhi 0.4)
WARMUP = 3.0         # v3 wala (100-epoch run ke liye)
PATIENCE = 20        # v3 wala (v2.2 ka patience 12 epoch 14 par ruk gaya tha — 20 rakho)

# ---- Starting weights: COCO yolov8s.pt (Ultralytics khud download karta hai) ----
# v4 FULL RETRAIN hai — kisi purane model se start NAHI hota. Drive se kuch nahi aata.
MODEL_PATH = "yolov8s.pt"

# ---- v3 baseline (frozen test) — adoption gate ka pehla hissa ----
# Notebook wala gate frozen test (680, leaked) par hai. ASLI gate local hai:
# cleaned test (511, mAP50 0.8378) + hard-frame eval + live retest — wo download ke baad Muse karega.
V3_BASELINE_MAP50 = 0.8666
V3_BASELINE_GUN_AP50 = 0.7839
V3_BASELINE_KNIFE_AP50 = 0.9493
V3_BASELINE_MAP50_CLEAN = 0.8378  # cleaned 511-image test — local gate ke liye record

DATASET_SLUG = "sentinel-weapon-v4"  # Add Input se attach hona chahiye

# Dataset ka path hardcode nahi hai — Step 2 usay /kaggle/input ke andar
# structure (images/ + labels/) se dhoondta hai, kyunki Kaggle ka mount path
# mukhtalif ho sakta hai (/kaggle/input/<slug>/ ya /kaggle/input/datasets/...).
# /kaggle/input read-only hai, is liye Step 5 dataset ko WORK mein copy karta hai.
WORK = Path("/kaggle/working/datasets")
RUNS_PROJECT = "/kaggle/working/runs/weapon"
OUTPUT_DIR = Path("/kaggle/working/output") / RUN_NAME   # best.pt + results yahan — run ke baad download kar lena

print("RUN_MODE :", RUN_MODE, "| EPOCHS:", EPOCHS)
print("RUN_NAME :", RUN_NAME)
print(f"Full retrain: lr0={LR0} warmup={WARMUP} degrees={DEGREES} perspective={PERSPECTIVE} close_mosaic={CLOSE_MOSAIC} mixup={MIXUP} erasing={ERASING} patience={PATIENCE} seed={SEED} deterministic=True")
print(f"New-files oversample: x{NEW_OVERSAMPLE} (band) | Knife oversample: x{OVERSAMPLE_KNIFE} (saare knife images)")
print("Starting weights: COCO yolov8s.pt (Ultralytics auto-download)")
print("DEVICE:", DEVICE, "| BATCH:", BATCH)


## Step 1 — GPU check

In [ ]:
!nvidia-smi
import torch
print("CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "GPU nahi mila — notebook settings mai Accelerator = GPU T4 x2 select karo, phir dobara chalao."
N_GPU = torch.cuda.device_count()
print("GPUs:", N_GPU, "|", [torch.cuda.get_device_name(i) for i in range(N_GPU)])
# Lambi run (100 epochs) 1 GPU par ~13 ghante leti hai — 12h session cap se zyada,
# is liye wahin fail ho jao bajaye 12 ghante quota jalane ke.
if EPOCHS > 25:
    assert N_GPU >= 2, "100-epoch run ke liye 2 GPUs chahiye (T4 x2) — 1 GPU par ye 12h cap se zyada legi. Notebook settings mai Accelerator = GPU T4 x2 select karo."
# DEVICE/BATCH CONFIG cell (cell 1) mein derive hote hain — yahan sirf check + print.
print("DEVICE:", DEVICE, "| BATCH:", BATCH)

## Step 2 — Dataset dhoondna (Add Input se `sentinel-weapon-v4` attach hona chahiye)


In [ ]:
# Kaggle datasets ko mukhtalif jagah mount kar sakta hai (/kaggle/input/<slug>/
# ya /kaggle/input/datasets/...), is liye path hardcode karne ki bajaye
# structure se dhoondo: wo folder jiske andar images/ aur labels/ dono hon.
INPUT_ROOT = Path("/kaggle/input")
cands_src = [
    p.parent
    for p in INPUT_ROOT.rglob("images")
    if p.is_dir() and (p.parent / "labels").is_dir()
]
assert cands_src, f"Dataset nahi mila: /kaggle/input ke andar images/+labels/ wala folder nahi — right sidebar mai Add Input se dataset '{DATASET_SLUG}' attach karo, phir dobara chalao."
# MUST-14: sirf wo candidates jinke path mein DATASET_SLUG ho — do datasets
# attach hon (v4 + v5) to shallowest-pick ghalat dataset par 7-8 ghante ki
# run karwa sakta hai. Exactly ek match hona chahiye.
cands_src = [p for p in cands_src if DATASET_SLUG in str(p)]
assert len(cands_src) == 1, f"Dataset '{DATASET_SLUG}' ka exactly ek match chahiye tha, {len(cands_src)} mile — Add Input mein sirf '{DATASET_SLUG}' attach rakho, phir dobara chalao."
DATASET_SRC = cands_src[0]
print("Dataset mila:", DATASET_SRC)

## Step 3 — Ultralytics install (starting weights COCO yolov8s.pt hain — Ultralytics khud download karta hai)


In [ ]:
%pip install -q ultralytics==8.4.215

# v4 FULL RETRAIN hai — starting weights COCO yolov8s.pt hain, jo Ultralytics
# training shuru hote hi khud download kar leta hai (MODEL_PATH = "yolov8s.pt").
# Drive se kuch download nahi hota is run mein (v4ft wala gdown step yahan nahi hai).
# MUST-11: ultralytics PINNED hai (requirements.txt wali 8.4.215) — unpinned
# install kisi nayi release par chala jata to defaults (e.g. fitness ki
# definition) chup-chaap badal sakte hain. Versions print + record karo.
import sys
import ultralytics
print(f"ultralytics: {ultralytics.__version__}")
print(f"torch: {torch.__version__} | cuda: {torch.version.cuda}")
print(f"python: {sys.version}")
print("Ultralytics installed. Starting weights: COCO yolov8s.pt (auto-download on first use).")
print("MODEL_PATH:", MODEL_PATH)


## Step 4 — Repo clone (training script wahi se ayega)

In [ ]:
%cd /kaggle/working
!rm -rf /kaggle/working/sentinel_repo
!git clone -b $REPO_BRANCH --single-branch https://github.com/muddassir1935/FYP.git /kaggle/working/sentinel_repo
%cd /kaggle/working/sentinel_repo
# MUST-12: REPO_COMMIT set ho to usi commit par pin karo (floating main nahi),
# phir jo commit bhi chala uska SHA record karo — training script ko
# --code-revision se pass hota hai aur run_summary.json mein aata hai.
if REPO_COMMIT:
    !git checkout $REPO_COMMIT
CODE_REVISION = !git rev-parse HEAD
CODE_REVISION = CODE_REVISION[0].strip()
if REPO_COMMIT:
    assert CODE_REVISION.startswith(REPO_COMMIT), f"REPO_COMMIT checkout fail: {REPO_COMMIT} chahiye tha, {CODE_REVISION} mila — SHA check karo, Muse ko batao."
print("CODE_REVISION:", CODE_REVISION)
%cd /kaggle/working

script = Path("/kaggle/working/sentinel_repo/scripts/training/train_weapon_detector.py")
assert script.exists(), "Training script nahi mila — branch ka naam CONFIG mai check karo, Muse ko batao."
print("Training script ready:", script)

## Step 5 — Dataset copy + verification (manifest se review status + counts check — UNREVIEWED ho ya counts galat hon to training yahi ruk jayegi)

In [ ]:
import json
import shutil

WORK.mkdir(parents=True, exist_ok=True)
# /kaggle/input read-only hai aur oversampling (Step 5b) ko likhna hota hai,
# is liye dataset ko WORK mein copy karo. Pehle se copy ho to fresh copy banao
# (purani copy mein oversample ki hui _os1 files bhi ho sakti hain).
dest = WORK / DATASET_SRC.name  # dataset ka apna naam (weapon_yolo_v4) — hardcode nahi
if dest.exists():
    print("Purani copy mili, fresh copy bana raha hun...")
    shutil.rmtree(dest)
# Poora dataset copy karne ki bajaye symlinks banao: Kaggle ka input mount
# chhoti files ka data copy karte hue bahut slow hai (v7 run isi copy par ruk
# gaya), jab ke training ko sirf parhna hota hai — symlinks se kaam chal jata
# hai. Oversampling phir bhi ASLI _os1 files likhta hai.
# NOTE: copytree(symlinks=True) sirf pehle se mojood symlinks ko bachata hai,
# naye symlinks NAHI banata — is liye copy_function=os.symlink zaroori hai.
import os
linked = [0]
def _link(src, dst):
    os.symlink(src, dst)
    linked[0] += 1
    if linked[0] % 5000 == 0:
        print(f"  ... {linked[0]} files linked", flush=True)
print(f"Linking (symlinks) {DATASET_SRC} -> {dest} ...")
shutil.copytree(DATASET_SRC, dest, copy_function=_link)
print(f"Linking ho gaya: {linked[0]} files.")

# dataset root = wo folder jiske andar images/ aur labels/ dono hon (zip ki structure dono surton mai handle hai)
# MUST-14: sirf is dataset ki copy (dest) ke andar dhoondo — WORK mein purani
# datasets ki copies hon to shallowest-pick ghalat root pakar sakta hai.
cands = [p.parent for p in dest.rglob("images") if (p.parent / "labels").exists()]
assert cands, "Dataset structure samajh nahi aya — ye output Muse ko bhejo."
root = sorted(cands, key=lambda p: len(p.parts))[0]
print("Dataset root:", root)

# ---- manifest check: review status + expected counts + pHash audit ----
manifest_path = root / "manifest.json"
assert manifest_path.exists(), "manifest.json nahi mila — ye dataset assembly script se nahi bana. Ye output Muse ko bhejo."
manifest = json.loads(manifest_path.read_text())
review_status = manifest.get("review_status")
print("Manifest review_status:", review_status)
assert review_status == "reviewed", "LABELS ABHI UNREVIEWED HAIN — user ka final review mukammal hone aur dataset dobara assemble hone se pehle training NAHI ho sakti. Ruk jao, Muse ko batao."
audit = manifest.get("audit", {})
print("pHash audit verdict:", audit.get("verdict"), f"(leakage: {audit.get('leakage_count')})")
assert audit.get("verdict") == "CLEAN", "pHash audit CLEAN nahi hai — frozen test mein leakage hai. Training roki gayi. Ye output Muse ko bhejo."
final = manifest.get("final", {})
print("Train breakdown:", json.dumps(final.get("train_breakdown", {})))

IMG_EXTS = {".jpg", ".jpeg", ".png"}
expected = {"train": final["train"], "val": final["val"], "test": final["test"]}
# val 1,351 aur test 680 v2 se FROZEN hain — in mein koi tabdeeli nahi honi chahiye
assert expected["val"] == 1351 and expected["test"] == 680, f"Val/test FROZEN counts galat hain (val {expected['val']}, test {expected['test']}) — ye output Muse ko bhejo."
ok = True
for split, exp in expected.items():
    imgs = [p for p in (root / "images" / split).glob("*") if p.suffix.lower() in IMG_EXTS]
    lbls = list((root / "labels" / split).glob("*.txt"))
    print(f"{split}: images = {len(imgs)} (expected {exp}), labels = {len(lbls)}")
    if len(imgs) != exp:
        ok = False

# SHOULD-6: train split mein har label ka image aur har image ka label hona
# chahiye — Ultralytics imageless labels ko chup-chaap ignore kar deta hai,
# to orphan labels ka matlab hai manifest se kam labels par training.
train_img_stems = {p.stem for p in (root / "images" / "train").glob("*") if p.suffix.lower() in IMG_EXTS}
train_lbl_stems = {p.stem for p in (root / "labels" / "train").glob("*.txt")}
assert train_lbl_stems == train_img_stems, (
    f"Train image/label mismatch: {len(train_lbl_stems - train_img_stems)} orphan labels, "
    f"{len(train_img_stems - train_lbl_stems)} images baghair label — dataset copy/symlink mein kharabi hai. Ye output Muse ko bhejo."
)

# labels sanity: har line ki pehli value (class id) sirf 0 (Gun) ya 1 (Knife) ho
# Poora check local machine par pehle se ho chuka hai (assembly script har
# label validate karta hai) aur Kaggle wali files wahi hain (same zip se
# extract hui), is liye yahan SAMPLE check kaafi hai — Kaggle ke slow mount
# par saari files parhna bahut slow hai (v9 run isi par atak gaya). Har 12th
# file check karo (~2,000 files, teeno splits se). Koi systematic kharabi
# hoti to sample mein pakri jati.
all_lbl = sorted((root / "labels").rglob("*.txt"))
sample = all_lbl[::12]
bad, checked = 0, 0
for i, txt in enumerate(sample, 1):
    if i % 500 == 0:
        print(f"  ... {i}/{len(sample)} sample label files checked ({checked} lines)", flush=True)
    for line in txt.read_text().splitlines():
        if line.strip():
            checked += 1
            if int(float(line.split()[0])) not in (0, 1):
                bad += 1
print(f"label lines checked = {checked} (sample {len(sample)}/{len(all_lbl)} files), bad class ids = {bad}")

assert ok and bad == 0, "DATASET VERIFICATION FAILED — training roki gayi. Ye poora output Muse ko bhejo."

# Kaggle ke liye data yaml (repo wali yaml local paths ke liye hai, ye wali /kaggle/working ke absolute path ke saath)
DATA_YAML = root / "sentinel_data.yaml"
DATA_YAML.write_text(f"path: {root}\ntrain: images/train\nval: images/val\ntest: images/test\nnc: 2\nnames: ['Gun', 'Knife']\n")
print("DATA_YAML:", DATA_YAML)
print("DATASET VERIFIED OK")


## Step 5b — Oversampling: Knife 2x (saare knife images) — naye files ka oversampling BAND

Do hisse thay v4ft mein; v4 mein sirf aik chalta hai:

1. **Naye files x1 (BAND)** (`NEW_OVERSAMPLE = 1`): Test8+Test9 ke 145 naye frames natural frequency par rehte hain — 100 epochs mein har frame 100 baar dekha jata hai, jo v4ft ke 80 exposures (8 epochs x 10 copies) ke barabar hai, baghair decision boundary shift kiye.
2. **Knife 2x** (`OVERSAMPLE_KNIFE = 2`): v3 wala setting — SAARE Knife images par lagta hai (naye Test9 knife positives sameet), taake Gun:Knife imbalance (5.65:1) kam ho.

Copies ASLI files hain (symlinks nahi) — v3 notebook wala pattern.


In [ ]:
# Step 5b — Oversampling: naye files x1 (band) + Knife 2x (saare knife images)
import shutil

train_img = root / "images" / "train"
train_lbl = root / "labels" / "train"

# NICE-8: pehle se bani _os label jiska image ghayab ho (adhoori copy se bachi
# hui) delete karo — warna `dst_lbl.exists()` check use permanent orphan bana
# deta hai, kyunki dobara chalne par copy skip ho jati hai.
for _orphan in train_lbl.glob("*_os*.txt"):
    if not any((train_img / (_orphan.stem + ext)).exists() for ext in (".jpg", ".jpeg", ".png")):
        print(f"  orphan _os label delete ki (image nahi mila): {_orphan.name}")
        _orphan.unlink()

# ---- Phase A: naye files x10 (new_files.txt se) ----
new_list = root / "new_files.txt"
assert new_list.exists(), "new_files.txt nahi mila — ye dataset assemble_v4_dataset.py se nahi bana. Ye output Muse ko bhejo."
new_stems = {l.strip() for l in new_list.read_text().splitlines() if l.strip()}
print(f"New files (new_files.txt): {len(new_stems)}")
assert NEW_OVERSAMPLE >= 1, "NEW_OVERSAMPLE kam az kam 1 hona chahiye (1 = oversampling band)"
made_nf, skipped_nf = 0, 0
if NEW_OVERSAMPLE == 1:
    print("New-files oversampling band hai (NEW_OVERSAMPLE = 1).")
else:
    for stem in sorted(new_stems):
        lbl = train_lbl / f"{stem}.txt"
        img = None
        for ext in (".jpg", ".jpeg", ".png"):
            cand = train_img / (stem + ext)
            if cand.exists():
                img = cand
                break
        if img is None or not lbl.exists():
            skipped_nf += 1
            continue
        for k in range(1, NEW_OVERSAMPLE):
            dst_lbl = train_lbl / f"{stem}_nf{k}.txt"
            if dst_lbl.exists():
                continue  # cell dobara chale to copies dobara na banein
            # NICE-8: pehle image, phir label — beech mein crash ho to orphan label na bane.
            shutil.copy2(img, train_img / f"{stem}_nf{k}{img.suffix}")
            shutil.copy2(lbl, dst_lbl)
            made_nf += 1
            if made_nf % 500 == 0:
                print(f"  ... {made_nf} new-file copies ban gayi", flush=True)
    print(f"New-file copies made = {made_nf}, skipped (no image/label) = {skipped_nf}")

# ---- Phase B: Knife 2x — SAARE knife images (v3 recipe; naye files sameet) ----
assert OVERSAMPLE_KNIFE >= 1, "OVERSAMPLE_KNIFE kam az kam 1 hona chahiye (1 = oversampling band)"
if OVERSAMPLE_KNIFE == 1:
    print("Knife oversampling band hai (OVERSAMPLE_KNIFE = 1).")
else:
    knife_imgs, made, skipped, scanned = 0, 0, 0, 0
    for lbl in sorted(train_lbl.glob("*.txt")):
        scanned += 1
        if scanned % 5000 == 0:
            print(f"  ... {scanned} train labels scanned, {knife_imgs} knife images mili", flush=True)
        if "_nf" in lbl.stem or lbl.stem.endswith("_os1"):
            continue  # pehle se bani copy — dobara duplicate mat karo
        has_knife = any(
            line.strip() and int(float(line.split()[0])) == 1
            for line in lbl.read_text().splitlines()
        )
        if not has_knife:
            continue
        knife_imgs += 1
        img = None
        for ext in (".jpg", ".jpeg", ".png"):
            cand = train_img / (lbl.stem + ext)
            if cand.exists():
                img = cand
                break
        if img is None:
            skipped += 1
            continue
        for k in range(1, OVERSAMPLE_KNIFE):
            dst_lbl = train_lbl / f"{lbl.stem}_os{k}.txt"
            if dst_lbl.exists():
                continue
            # NICE-8: pehle image, phir label — beech mein crash ho to orphan label na bane.
            shutil.copy2(img, train_img / f"{lbl.stem}_os{k}{img.suffix}")
            shutil.copy2(lbl, dst_lbl)
            made += 1
            if made % 500 == 0:
                print(f"  ... {made} knife copies ban gayi", flush=True)
    print(f"knife images found (all) = {knife_imgs}, copies made = {made}, skipped (no image) = {skipped}")

total = len([p for p in train_img.glob("*") if p.suffix.lower() in IMG_EXTS])
print(f"train images ab = {total} (oversampling se pehle {final['train']})")

# SHOULD-6 (dobara): oversampling ke baad bhi train image/label stems barabar hon.
train_img_stems = {p.stem for p in train_img.glob("*") if p.suffix.lower() in IMG_EXTS}
train_lbl_stems = {p.stem for p in train_lbl.glob("*.txt")}
assert train_lbl_stems == train_img_stems, (
    f"Oversampling ke baad train image/label mismatch: {len(train_lbl_stems - train_img_stems)} orphan labels, "
    f"{len(train_img_stems - train_lbl_stems)} images baghair label — ye output Muse ko bhejo."
)
print("OVERSAMPLING DONE")


## Step 6 — Full-retrain training (v4, 100 epochs, 2x T4 par andazan ~7-8 ghante — Save & Run All se chalao)


In [ ]:
%cd /kaggle/working/sentinel_repo
print("Full-retrain start: COCO yolov8s -> v4 mix | "
      f"epochs={EPOCHS} lr0={LR0} warmup={WARMUP} degrees={DEGREES} perspective={PERSPECTIVE} "
      f"close_mosaic={CLOSE_MOSAIC} mixup={MIXUP} erasing={ERASING} patience={PATIENCE} "
      f"seed={SEED} deterministic=True "
      f"| batch={BATCH} device={DEVICE} (Ultralytics har epoch ki progress khud print karega)", flush=True)
!python scripts/training/train_weapon_detector.py --data $DATA_YAML --model $MODEL_PATH --epochs $EPOCHS --imgsz 640 --batch $BATCH --device $DEVICE --workers 4 --patience $PATIENCE --seed $SEED --deterministic --lr0 $LR0 --warmup-epochs $WARMUP --degrees $DEGREES --perspective $PERSPECTIVE --close-mosaic $CLOSE_MOSAIC --mixup $MIXUP --erasing $ERASING --code-revision $CODE_REVISION --dataset-slug $DATASET_SLUG --project $RUNS_PROJECT --name $RUN_NAME
print("Training cell khatam — agla cell (Step 7) test metrics dikhayega.", flush=True)


## Step 7 — Results (frozen test per final numbers + v3 baseline se comparison + adoption gate ka pehla hissa)


In [ ]:
import json
from IPython.display import Image, display

run_dir = Path(RUNS_PROJECT) / RUN_NAME
summary = json.loads((run_dir / "run_summary.json").read_text())
print("TEST METRICS:", json.dumps(summary["test_metrics"], indent=2))

# v3 baseline se comparison — adoption gate ka PEHLA hissa (notebook wala gate
# frozen test par hai; ASLI faisla cleaned test + hard-frame eval + live retest
# par hota hai, jo download ke baad Muse local karta hai).
tm = summary["test_metrics"]
new_map50 = tm.get("mAP50")
print(f"\nBaseline v3 (frozen test): mAP50 = {V3_BASELINE_MAP50} | Gun AP50 = {V3_BASELINE_GUN_AP50} | Knife AP50 = {V3_BASELINE_KNIFE_AP50}")
print(f"Full retrain v4 (frozen test): mAP50 = {new_map50} | " + " | ".join(f"{k} AP50 = {v.get('AP50')}" for k, v in tm.get("per_class", {}).items()))
print(f"Cleaned-test reference (local gate): v3 = {V3_BASELINE_MAP50_CLEAN} (511 images)")
if new_map50 is not None and new_map50 > V3_BASELINE_MAP50:
    print(f"GATE (frozen test): PASS — v4 beats v3 by {new_map50 - V3_BASELINE_MAP50:.4f} mAP50. Ab cleaned-test + hard-frame + live retest hoga — wo pass ho to adopt.")
elif new_map50 is not None:
    print(f"GATE (frozen test): FAIL — v4 ({new_map50}) v3 baseline ({V3_BASELINE_MAP50}) se neeche hai. v3 weights hi fallback rehte hain. Run log mein darj karo aur Muse ko batao.")
else:
    print("GATE: UNKNOWN — test metrics nahi mile. Ye output Muse ko bhejo.")

cm = run_dir / "confusion_matrix.png"
if cm.exists():
    display(Image(str(cm), width=640))


## Step 8 — Weights + results output folder mein save (run ke baad download kar lena + COPY block Muse ko bhejna hai)

In [ ]:
import hashlib
import shutil

dest = OUTPUT_DIR
dest.mkdir(parents=True, exist_ok=True)

best = run_dir / "weights" / "best.pt"
# NICE-7: best.pt na ho to clear message do — shutil.copy ka traceback nahi.
assert best.exists(), f"best.pt nahi mila: {best} — training fail hui ya run dir ghalat hai. Ye output Muse ko bhejo."
dated = dest / f"{RUN_NAME}_best.pt"
shutil.copy(best, dated)
# MUST-13: args.yaml = full effective config (defaults sameet) — v3/v4 mein ye
# download hi nahi hua tha, is liye un runs ke defaults ka koi record nahi hai.
for extra in ["results.csv", "confusion_matrix.png", "results.png", "run_summary.json", "args.yaml"]:
    srcp = run_dir / extra
    if srcp.exists():
        shutil.copy(srcp, dest / extra)
# pip freeze snapshot — kaunsi library versions par run hua, iska record.
import subprocess
with open(dest / "pip_freeze.txt", "w") as _fh:
    subprocess.run(["pip", "freeze"], stdout=_fh, check=False)
# last.pt bhi rakh lo — agar kabhi resume karna pare to kaam ayega
last = run_dir / "weights" / "last.pt"
if last.exists():
    shutil.copy(last, dest / f"{RUN_NAME}_last.pt")

sha = hashlib.sha256(dated.read_bytes()).hexdigest()
tm = summary["test_metrics"]
print("=" * 60)
print("===== COPY FROM HERE — ye block Muse ko bhej dena =====")
print(f"RUN OK — {RUN_NAME}")
print(f"Weights file : {dated.name}")
print(f"Size (MB)    : {dated.stat().st_size / 1e6:.2f}")
print(f"SHA-256      : {sha}")
print(f"Test mAP@0.5 : {tm.get('mAP50')} | mAP@0.5:0.95 : {tm.get('mAP50_95')} | P : {tm.get('precision')} | R : {tm.get('recall')} | F1 : {tm.get('f1')}")
print(f"vs v3 baseline mAP@0.5 (frozen test) : {V3_BASELINE_MAP50} | Gate: {'PASS' if (tm.get('mAP50') or 0) > V3_BASELINE_MAP50 else 'FAIL — v3 weights stay the fallback'}")
for cls, vals in tm.get("per_class", {}).items():
    print(f"  {cls}: AP50 = {vals.get('AP50')}")
print("===== COPY TO HERE =====")
print("=" * 60)
print(f"Weights + results saved in: {dest}  (Kaggle Output tab se download kar lo)")